# 🐝 Lab 3.1 — A Modern GPT From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab07_gpt_from_scratch.ipynb)

**⏱ Time:** ~2.5 hours  ·  **Day 3 — Transformers & Beyond**

Buzz the Backprop Bee has read every play in the hive library and now wants to *write* one. Today we build the brain for that:
a small, **Llama-style** decoder-only Transformer, one Lego brick at a time, then train it on Tiny Shakespeare until it babbles in iambic-ish English.

### 🎯 Learning objectives
By the end of this lab you will be able to:
- Implement **scaled dot-product attention** and a **causal mask** from scratch, and verify them against `F.scaled_dot_product_attention`.
- Build **multi-head attention** with efficient reshapes, and check it against `nn.MultiheadAttention`.
- Explain and implement **Rotary Position Embeddings (RoPE)**, and show numerically that they encode *relative* position.
- Assemble a modern block (**RMSNorm, SwiGLU, pre-norm residuals, weight tying**) and say why each piece replaced its 2017 ancestor.
- Train it with **AdamW + warmup/cosine, gradient clipping and mixed precision**, then look inside with **attention heatmaps**.

### 🧰 Prerequisites
Comfortable with PyTorch `nn.Module`s, autograd and a basic training loop; matrix shapes and broadcasting; softmax and cross-entropy.

### 🌍 Why this matters in 2026
Most open-weight LLMs you will meet (Llama, Qwen, Mistral, Gemma, DeepSeek...) are variations on the block you build today: decoder-only, RMSNorm, pre-norm residuals, a gated MLP (SwiGLU or a close cousin such as GeGLU), RoPE, and attention. The frontier models differ in *scale* and in tweaks (grouped-query or latent attention, mixture-of-experts MLPs, hybrid layers), but if you can write this ~150-line model from memory you can read almost any modern architecture's source code and know exactly where the interesting changes are. Labs 3.2 and 3.3 build directly on this one.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 · Setup | Imports, seeds, device | 5 |
| 1 · Text → tokens | Character tokenizer, batches, the unigram baseline | 15 |
| 2 · Attention & causal mask | Implement attention, see why we scale by √d | 25 |
| 3 · Multi-head attention | Heads as a reshape; match `nn.MultiheadAttention` | 15 |
| 4 · RoPE | Rotate q and k; prove scores depend on relative position | 25 |
| 5 · The modern block | RMSNorm, SwiGLU, pre-norm, weight tying | 20 |
| 6 · Assemble & train | AdamW, warmup + cosine, clipping, AMP; watch it learn to write | 30 |
| 7 · Generation & a look inside | Sample text, attention heatmaps, head probes | 15 |
| Wrap-up | Takeaways, interview corner, boss level | 5 |

## 0 · Setup 🛠️
Everything here runs on CPU in ~10–15 minutes. On a Colab GPU (Runtime → Change runtime type → T4) the notebook automatically uses a bigger model and mixed precision.

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=1337):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})

from dataclasses import dataclass

torch.set_float32_matmul_precision("high")
# No %pip installs needed: everything used here ships with Colab (tiktoken is optional and guarded).
print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")

## Part 1 · Text → tokens 🔤

A language model never sees letters; it sees **integers**. A *tokenizer* is the bijection between strings and integer sequences.

* **Character-level** (what we use today): vocabulary = the set of distinct characters (~65 for Shakespeare). Tiny vocab, long sequences, zero preprocessing magic.
* **Byte-Pair Encoding (BPE)** (what real LLMs use): start from bytes, repeatedly merge the most frequent adjacent pair into a new token. GPT-2 has 50,257 tokens; modern models use 100k–200k+. Roughly 4 characters per token in English, so the *same* context window holds ~4× more text.

The model's job is next-token prediction: given tokens $x_1, \dots, x_t$, output a distribution $p_\theta(x_{t+1} \mid x_{\le t})$ and minimise the average negative log-likelihood (cross-entropy)
$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t}).$$

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_DIR = os.environ.get("LAB_DATA_DIR", os.path.join(os.path.expanduser("~"), ".cache", "gradient_hive"))
os.makedirs(DATA_DIR, exist_ok=True)

# A small embedded fallback (public-domain Shakespeare, Coriolanus I.i) in case the download fails.
FALLBACK = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them. Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.

"""

def load_tiny_shakespeare():
    path = os.path.join(DATA_DIR, "tinyshakespeare.txt")
    if os.path.exists(path):
        return open(path, encoding="utf-8").read()
    try:
        import urllib.request
        with urllib.request.urlopen(DATA_URL, timeout=30) as r:
            text = r.read().decode("utf-8")
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
        return text
    except Exception as e:
        print("⚠️  Download failed, using the small embedded fallback text:", repr(e))
        return FALLBACK * 60

text = load_tiny_shakespeare()
print(f"{len(text):,} characters")
print(text[:300])

In [ ]:
class CharTokenizer:
    """The simplest possible tokenizer: one token per distinct character."""
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.itos = {i: c for c, i in self.stoi.items()}
        self.vocab_size = len(self.chars)
    def encode(self, s):
        return [self.stoi[c] for c in s]
    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

tok = CharTokenizer(text)
print("vocab size:", tok.vocab_size)
print("vocab:", repr("".join(tok.chars)))
ids = tok.encode("To bee, or not to bee?")
print(ids, "->", repr(tok.decode(ids)))
assert tok.decode(tok.encode(text[:1000])) == text[:1000]

data = torch.tensor(tok.encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"train tokens: {len(train_data):,} | val tokens: {len(val_data):,}")

In [ ]:
# How skewed is the character distribution? (A unigram model's loss = its entropy.)
counts = torch.bincount(data, minlength=tok.vocab_size).float()
probs = counts / counts.sum()
order = probs.argsort(descending=True)
unigram_entropy = -(probs[probs > 0] * probs[probs > 0].log()).sum().item()

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(tok.vocab_size), probs[order].numpy(), color=HIVE[0], edgecolor=HIVE[1], linewidth=0.3)
ax.set_xticks(range(tok.vocab_size))
ax.set_xticklabels([repr(tok.itos[int(i)])[1:-1] for i in order], fontsize=7)
ax.set_ylabel("frequency"); ax.set_title("Tiny Shakespeare character frequencies")
plt.show()
print(f"uniform-guess loss  ln(V) = {math.log(tok.vocab_size):.3f} nats")
print(f"unigram (no context) loss = {unigram_entropy:.3f} nats   <- any model that uses context should beat this")

**Optional peek at BPE.** If `tiktoken` is available (it is on most Colab images), compare how GPT-2's BPE splits a line vs our character tokenizer. Fewer tokens per line = more text per context window, but a much larger embedding table (50k × d instead of 65 × d), which is why we stay character-level on CPU.

In [ ]:
line = "O Romeo, Romeo! wherefore art thou Romeo?"
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    bpe = enc.encode(line)
    print(f"GPT-2 BPE: {len(bpe)} tokens ->", [enc.decode([t]) for t in bpe])
except Exception as e:
    print("tiktoken not available (that's fine):", type(e).__name__)
print(f"char-level: {len(tok.encode(line))} tokens")

In [ ]:
def get_batch(split, batch_size, block_size, device=DEVICE):
    """Sample random windows. y is x shifted left by one: predict the next char at every position."""
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", 2, 16)
for b in range(2):
    print("x:", repr(tok.decode(xb[b])), "\ny:", repr(tok.decode(yb[b])))

Notice that **one window of length $T$ gives $T$ training examples**: predict `y[0]` from `x[:1]`, `y[1]` from `x[:2]`, and so on. The causal mask (Part 2) is what lets us train on all $T$ of them in one parallel forward pass.

> **🍯 Key takeaway:** A language model only ever sees integers. One window of T tokens gives T next-token training examples, and the unigram entropy is the bar any model that uses context must beat.

## Part 2 · Scaled dot-product attention & the causal mask 🎯

**Intuition.** Each position emits three vectors:
a **query** $q$ ("what am I looking for?"), a **key** $k$ ("what do I contain?") and a **value** $v$ ("what do I hand over if you pick me?").
Position $i$ scores every position $j$ by $q_i \cdot k_j$, turns the scores into a probability distribution with softmax, and returns the weighted average of the values. It is a *soft, differentiable dictionary lookup*.

$$\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V,
\qquad M_{ij} = \begin{cases}0 & j \le i\\ -\infty & j > i\end{cases}$$

**Why divide by $\sqrt{d_k}$?** If the entries of $q$ and $k$ are independent with mean 0 and variance 1, then $q\cdot k = \sum_{m=1}^{d_k} q_m k_m$ has variance $d_k$. Without scaling, logits grow like $\sqrt{d_k}$, softmax saturates to a near one-hot, and its gradients vanish. Dividing by $\sqrt{d_k}$ keeps the logits at unit variance regardless of head size. Let's *see* it:

In [ ]:
dims = [4, 16, 64, 256, 1024]
T_demo = 64
res = {"unscaled": [], "scaled": []}
for d in dims:
    q = torch.randn(2000, d); k = torch.randn(2000, T_demo, d)
    logits = torch.einsum("nd,ntd->nt", q, k)
    res["unscaled"].append(logits.softmax(-1).max(-1).values.mean().item())
    res["scaled"].append((logits / math.sqrt(d)).softmax(-1).max(-1).values.mean().item())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(dims, res["unscaled"], "o-", label=r"softmax($q\cdot k$)")
ax.plot(dims, res["scaled"], "s-", label=r"softmax($q\cdot k/\sqrt{d}$)")
ax.axhline(1 / T_demo, ls=":", color="gray", label="uniform (1/T)")
ax.set_xscale("log", base=2); ax.set_xlabel("head dimension $d_k$"); ax.set_ylabel("mean max attention weight")
ax.set_title("Without scaling, attention collapses to one-hot as $d_k$ grows"); ax.legend()
plt.show()

### 🧪 Your Turn #1 — the causal mask
A decoder must not peek at the future: position $i$ may only attend to positions $j \le i$.
Write `causal_mask(T)` returning a **boolean** `(T, T)` tensor that is `True` where attention is **allowed** (this is the convention `F.scaled_dot_product_attention` uses for boolean masks).

*Hints:* `torch.tril` on a matrix of ones gives the lower triangle. Make sure the dtype is `torch.bool` and it lands on the requested `device`.

In [ ]:
def causal_mask(T, device=None):
    """Return a (T, T) bool tensor, True where query i may attend to key j (i.e. j <= i)."""
    # TODO: build a lower-triangular boolean matrix
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def causal_mask(T, device=None):
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))

### 🧪 Your Turn #2 — scaled dot-product attention
Implement `attention(q, k, v, mask)` for tensors of shape `(..., T, d)` (the `...` will be `batch, heads`). Return **both** the output `(..., T, d)` and the attention weights `(..., T, T)` (we'll plot them later).

*Hints:* `scores = q @ k.transpose(-2, -1) / sqrt(d)`; use `scores.masked_fill(~mask, float("-inf"))`; softmax over the **last** dim (over keys).

In [ ]:
def attention(q, k, v, mask=None):
    """q, k, v: (..., T, d). mask: bool (T, T), True = allowed. Returns (out, weights)."""
    d = q.shape[-1]
    # TODO 1: scores = ...
    # TODO 2: apply the mask (if given)
    # TODO 3: weights = softmax(...)  ;  out = weights @ v
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def attention(q, k, v, mask=None):
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = scores.softmax(dim=-1)
    return weights @ v, weights

### ✔️ Checkpoint — match PyTorch's fused kernel

In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(2, 3, 10, 16) for _ in range(3))
m = causal_mask(10)
assert m.dtype == torch.bool and m.shape == (10, 10) and m[0, 1] == False and m[1, 0] == True

out, w = attention(q, k, v, m)
ref_causal = F.scaled_dot_product_attention(q, k, v, is_causal=True)
ref_mask   = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
assert torch.allclose(out, ref_causal, atol=1e-5), "doesn't match is_causal=True"
assert torch.allclose(out, ref_mask, atol=1e-5), "doesn't match attn_mask=..."
assert torch.allclose(w.sum(-1), torch.ones(2, 3, 10), atol=1e-5), "rows of weights must sum to 1"
assert (w.triu(1) == 0).all(), "future positions must get exactly zero weight"
print("✔️ attention matches F.scaled_dot_product_attention — bzzzt, nice work!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(m.numpy(), cmap="YlOrBr"); axes[0].set_title("causal mask (True = allowed)")
im = axes[1].imshow(w[0, 0].numpy(), cmap="YlOrBr", vmin=0)
axes[1].set_title("attention weights (random q, k)")
for a in axes: a.set_xlabel("key position j"); a.set_ylabel("query position i")
plt.colorbar(im, ax=axes[1], fraction=0.046); plt.tight_layout(); plt.show()

Row $i$ of the right plot is the distribution query $i$ places over keys $0..i$. Row 0 is forced to put all its mass on itself. With *random* vectors the pattern is noise; after training (Part 7) the heads will learn structured patterns.

> **🍯 Key takeaway:** Attention is a soft dictionary lookup. The √d scaling keeps softmax out of saturation, and the causal mask is what lets one forward pass train on every position at once.

## Part 3 · Multi-head attention 🐝🐝🐝🐝

One attention distribution per position is a bottleneck: a token may want to look at the previous character *and* at the start of the current word *and* at the speaker's name. **Multi-head attention** runs $H$ attentions in parallel on $H$ lower-dimensional projections ($d_{head} = d_{model}/H$) and concatenates the results:

$$\operatorname{MHA}(X) = \operatorname{Concat}(\text{head}_1, \dots, \text{head}_H)\,W^O,\qquad \text{head}_h = \operatorname{Attention}(XW^Q_h, XW^K_h, XW^V_h).$$

Implementation trick: never loop over heads. Do **one** big `Linear(d, 3d)` for Q, K, V, then reshape `(B, T, 3d) → (B, T, 3, H, d_head) → 3 × (B, H, T, d_head)`, so the heads become just another batch dimension.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Plain causal MHA (no RoPE yet), using the attention() we just wrote."""
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.d_head).permute(2, 0, 3, 1, 4)  # each (B, H, T, dh)
        y, _ = attention(q, k, v, causal_mask(T, x.device))
        y = y.transpose(1, 2).reshape(B, T, C)   # concatenate heads
        return self.proj(y)

mha = MultiHeadAttention(32, 4)
print(mha(torch.randn(2, 7, 32)).shape)

### ✔️ Checkpoint — same numbers as `nn.MultiheadAttention`
Copy our weights into PyTorch's module and compare. (`nn.MultiheadAttention` uses the *opposite* boolean convention: `True` = **blocked**.)

In [ ]:
torch.manual_seed(0)
d, H, T = 32, 4, 9
ours = MultiHeadAttention(d, H)
ref = nn.MultiheadAttention(d, H, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(ours.qkv.weight)      # rows ordered [W_q; W_k; W_v], same as our chunking
    ref.out_proj.weight.copy_(ours.proj.weight)
x = torch.randn(3, T, d)
y_ref, _ = ref(x, x, x, attn_mask=~causal_mask(T), need_weights=False)
assert torch.allclose(ours(x), y_ref, atol=1e-5)
print("✔️ our MHA == nn.MultiheadAttention")

> **🍯 Key takeaway:** Multi-head attention is just a reshape: one big QKV projection, heads become a batch dimension, then concatenate and project.

## Part 4 · Rotary Position Embeddings (RoPE) 🌀

Attention is **permutation-equivariant**: shuffle the input tokens and the outputs shuffle the same way. Without positional information "dog bites man" = "man bites dog". The 2017 Transformer *added* sinusoidal vectors to the embeddings; GPT-2 learned an absolute position table. Modern LLMs use **RoPE** (Su et al., 2021): *rotate* queries and keys by an angle proportional to their position.

**The math.** Split a head vector into $d/2$ pairs $(x_{2i}, x_{2i+1})$. Pair $i$ gets frequency $\theta_i = b^{-2i/d}$ (base $b = 10{,}000$). At position $m$, rotate each pair by angle $m\theta_i$:
$$\begin{pmatrix}x'_{2i}\\x'_{2i+1}\end{pmatrix} = \begin{pmatrix}\cos m\theta_i & -\sin m\theta_i\\ \sin m\theta_i & \cos m\theta_i\end{pmatrix}\begin{pmatrix}x_{2i}\\x_{2i+1}\end{pmatrix}.$$

**Why it's brilliant.** Rotations compose: $R_m^\top R_n = R_{n-m}$. So
$$\langle R_m q,\; R_n k\rangle = q^\top R_m^\top R_n k = \langle q,\; R_{n-m}\,k\rangle,$$
the attention score depends only on the **relative offset** $n-m$, not on absolute positions, and no parameters are added. Fast-rotating pairs (small $i$) resolve nearby positions; slow pairs (large $i$) act like a coarse clock for long distances. Context-extension tricks such as Position Interpolation, NTK-aware scaling and YaRN all work by changing these frequencies.

> **Convention alert:** we pair *interleaved* dims `(0,1), (2,3), ...` as in the RoFormer paper and Meta's original Llama code. Hugging Face's `rotate_half` pairs `(i, i + d/2)` instead. Both are valid; they differ by a fixed permutation of the weights, which is why checkpoint conversion scripts permute Q/K.

In [ ]:
def rope_angles(T, d_head, base=10000.0, device=None):
    """cos/sin tables of shape (T, d_head/2): angle[m, i] = m * base^(-2i/d)."""
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    ang = torch.outer(torch.arange(T, device=device).float(), inv_freq)
    return ang.cos(), ang.sin()

cos, sin = rope_angles(128, 32)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
im = axes[0].imshow(cos.T.numpy(), aspect="auto", cmap="YlOrBr")
axes[0].set_xlabel("position m"); axes[0].set_ylabel("pair index i"); axes[0].set_title(r"$\cos(m\theta_i)$: fast pairs on top, slow at the bottom")
plt.colorbar(im, ax=axes[0])
for i, c in zip([0, 4, 15], [HIVE[2], HIVE[3], HIVE[4]]):
    th = 10000 ** (-2 * i / 32)
    for m_ in range(0, 8):
        a = m_ * th
        axes[1].arrow(0, 0, math.cos(a), math.sin(a), color=c, alpha=0.25 + 0.09 * m_, head_width=0.04, length_includes_head=True)
    axes[1].plot([], [], color=c, label=f"pair i={i}  (θ={th:.3g} rad/pos)")
axes[1].set_aspect("equal"); axes[1].set_xlim(-1.2, 1.2); axes[1].set_ylim(-1.2, 1.2)
axes[1].set_title("vector (1, 0) rotated at positions 0..7"); axes[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

### 🧪 Your Turn #3 — implement `apply_rope`
`x` has shape `(..., T, d_head)`; `cos`, `sin` have shape `(T_max, d_head/2)` with `T_max >= T`.

1. Slice the tables to the first `T` positions.
2. Split `x` into even dims `x1 = x[..., 0::2]` and odd dims `x2 = x[..., 1::2]`.
3. Rotate: `x1' = x1·cos − x2·sin`, `x2' = x1·sin + x2·cos`.
4. Re-interleave: `torch.stack([x1', x2'], dim=-1).flatten(-2)`.

*Hint:* broadcasting handles the leading `(B, H)` dims automatically because the tables are `(T, d/2)`.

In [ ]:
def apply_rope(x, cos, sin):
    """Rotate interleaved pairs of x (..., T, d) by position-dependent angles."""
    T = x.shape[-2]
    # TODO: slice cos/sin to T, split even/odd dims, rotate, re-interleave
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def apply_rope(x, cos, sin):
    T = x.shape[-2]
    cos, sin = cos[:T], sin[:T]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

### ✔️ Checkpoint — rotation, relativity, and agreement with the complex-number formulation
Meta's original Llama code implements RoPE by viewing each pair as a complex number $x_{2i} + i\,x_{2i+1}$ and multiplying by $e^{i m\theta_i}$. Let's check we agree.

In [ ]:
torch.manual_seed(0)
T, dh = 50, 16
cos, sin = rope_angles(T, dh)
x = torch.randn(2, 3, T, dh)
xr = apply_rope(x, cos, sin)

# 1) rotations preserve norms
assert torch.allclose(xr.norm(dim=-1), x.norm(dim=-1), atol=1e-5)
# 2) position 0 is unchanged
assert torch.allclose(xr[..., 0, :], x[..., 0, :], atol=1e-6)
# 3) complex-number reference
freqs = torch.polar(torch.ones_like(cos), torch.atan2(sin, cos))       # e^{i m theta}
xc = torch.view_as_complex(x.reshape(*x.shape[:-1], dh // 2, 2).contiguous())
ref = torch.view_as_real(xc * freqs).flatten(-2)
assert torch.allclose(xr, ref, atol=1e-5)
# 4) relative-position property: score(q@m, k@n) == score(q@m+s, k@n+s)
q, k = torch.randn(dh), torch.randn(dh)
def score_at(m, n):
    Q = torch.zeros(T, dh); K = torch.zeros(T, dh); Q[m] = q; K[n] = k
    return (apply_rope(Q, cos, sin)[m] @ apply_rope(K, cos, sin)[n]).item()
assert abs(score_at(10, 3) - score_at(30, 23)) < 1e-4 and abs(score_at(5, 5) - score_at(40, 40)) < 1e-4
print("✔️ RoPE: norm-preserving, matches complex formulation, and depends only on relative offset 🌀")

In [ ]:
# RoPE's "long-term decay": for q = k = all-ones, the score shrinks (noisily) as |n - m| grows.
dh, Tmax = 64, 1024
cos, sin = rope_angles(Tmax, dh)
ones = torch.ones(Tmax, dh)
rot = apply_rope(ones, cos, sin)
scores = (rot[0:1] @ rot.T).squeeze(0) / dh
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(scores.numpy(), lw=1)
ax.set_xlabel("relative distance n − m"); ax.set_ylabel("normalised q·k")
ax.set_title("RoPE score vs relative distance (q = k = 1): nearby tokens are favoured")
plt.show()

> **🍯 Key takeaway:** RoPE rotates q and k by position-dependent angles, so attention scores depend only on the relative offset, with zero extra parameters.

## Part 5 · The modern block: RMSNorm, SwiGLU, pre-norm, weight tying 🧱

| 2017 / GPT-2 | 2023+ (Llama-style) | Why the switch |
|---|---|---|
| LayerNorm (center + scale + bias) | **RMSNorm** (scale only) | ~Same quality, cheaper; no mean subtraction |
| Post-norm `x = LN(x + f(x))` | **Pre-norm** `x = x + f(Norm(x))` | Clean identity path for gradients; deep stacks train stably and are far less sensitive to warmup |
| ReLU/GELU MLP, 4× width | **SwiGLU** gated MLP, ~8/3× width | Gating consistently lowers loss at equal parameter count |
| Learned absolute positions | **RoPE** (Part 4) | Relative, parameter-free, extendable |
| Biases everywhere | **No biases** in linear layers | Fewer params, no quality loss, simpler |

**RMSNorm** (Zhang & Sennrich, 2019): $\operatorname{RMSNorm}(x) = \dfrac{x}{\sqrt{\frac{1}{d}\sum_j x_j^2 + \epsilon}} \odot g$, with a learned gain $g \in \mathbb{R}^d$.

**SwiGLU** (Shazeer, 2020): $\operatorname{MLP}(x) = W_{down}\big(\operatorname{SiLU}(W_{gate}x) \odot W_{up}x\big)$, where $\operatorname{SiLU}(z) = z\,\sigma(z)$. Three matrices instead of two, so the hidden width is shrunk from $4d$ to about $\tfrac{8}{3}d$ to keep the parameter count equal.

**Weight tying** (Press & Wolf, 2017): the output projection $W_{lm} \in \mathbb{R}^{V\times d}$ reuses the input embedding matrix. It saves $V\cdot d$ parameters (huge for 150k-token vocabularies) and often helps small models. Many large models (e.g. Llama 2/3 7–8B and up) leave them *untied*; small ones (e.g. Llama 3.2 1B/3B, Gemma) tie.

In [ ]:
z = torch.linspace(-5, 5, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(z, F.relu(z), label="ReLU"); ax.plot(z, F.gelu(z), label="GELU"); ax.plot(z, F.silu(z), label="SiLU / Swish")
ax.axhline(0, color="gray", lw=0.5); ax.legend(); ax.set_title("Smooth activations: SiLU is the 'S' in SwiGLU")
plt.show()

### 🧪 Your Turn #4 — RMSNorm
Implement `forward`. For mixed-precision safety, compute the mean of squares in `float32` and cast the result back to the input dtype (`.type_as(x)`) **before** multiplying by the gain.

*Hint:* `torch.rsqrt(t)` computes $1/\sqrt{t}$.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        # TODO: x / sqrt(mean(x^2) + eps) * weight   (compute in float32, cast back)
        raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        normed = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return normed.type_as(x) * self.weight

### 🧪 Your Turn #5 — SwiGLU MLP
Three bias-free linears: `w_gate: d→h`, `w_up: d→h`, `w_down: h→d`. Forward: `w_down(silu(w_gate(x)) * w_up(x))`.
The hidden size is `h = multiple_of * ceil((8/3·d) / multiple_of)` (rounding up to a hardware-friendly multiple, as Llama does).

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d, multiple_of=32, dropout=0.0):
        super().__init__()
        h = multiple_of * math.ceil(int(8 * d / 3) / multiple_of)
        # TODO: define self.w_gate, self.w_up, self.w_down (bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
class SwiGLU(nn.Module):
    def __init__(self, d, multiple_of=32, dropout=0.0):
        super().__init__()
        h = multiple_of * math.ceil(int(8 * d / 3) / multiple_of)
        self.w_gate = nn.Linear(d, h, bias=False)
        self.w_up = nn.Linear(d, h, bias=False)
        self.w_down = nn.Linear(h, d, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

### ✔️ Checkpoint

In [ ]:
torch.manual_seed(0)
x = torch.randn(4, 10, 48) * 3 + 1
rn = RMSNorm(48)
y = rn(x)
ref = x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)
assert torch.allclose(y, ref, atol=1e-5)
if hasattr(nn, "RMSNorm"):   # PyTorch >= 2.4 ships one
    assert torch.allclose(y, nn.RMSNorm(48, eps=1e-6)(x), atol=1e-5)
assert torch.allclose(y.pow(2).mean(-1), torch.ones(4, 10), atol=1e-3), "RMS of output should be ~1"

mlp = SwiGLU(48)
assert mlp.w_gate.weight.shape == (128, 48) and mlp.w_down.weight.shape == (48, 128)
assert torch.allclose(mlp(x), mlp.w_down(F.silu(mlp.w_gate(x)) * mlp.w_up(x)))
p_swiglu = sum(p.numel() for p in mlp.parameters()); p_classic = 2 * 48 * 4 * 48
print(f"SwiGLU params {p_swiglu:,} vs classic 4x MLP {p_classic:,} (≈ equal by design)")
print("✔️ RMSNorm and SwiGLU check out")

> **🍯 Key takeaway:** The 2023+ block is pre-norm + RMSNorm + SwiGLU + no biases. Each swap is small; together they make deep stacks train smoothly.

## Part 6 · Assemble the Llama-style GPT and train it 🏗️

```
tokens ─► Embedding ─► [ x = x + Attn(RMSNorm(x))      ] × L ─► RMSNorm ─► Linear (tied to Embedding) ─► logits
                        [ x = x + SwiGLU(RMSNorm(x))    ]
```
The **residual stream** is the highway; each sub-layer *reads* a normalised copy and *writes* an update back. Two initialisation details matter: all weights $\sim\mathcal{N}(0, 0.02^2)$, and the projections that write into the residual stream (`attn.proj`, `mlp.w_down`) are scaled down by $1/\sqrt{2L}$ (GPT-2's trick) so the stream's variance doesn't grow with depth.

Inside attention we use the fused `F.scaled_dot_product_attention` for speed (on GPU it dispatches to FlashAttention-style kernels; see Lab 3.2), but keep a switch that routes through *our* `attention()` so we can record the weights for plots.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    d_model: int = 128
    dropout: float = 0.0
    rope_base: float = 10000.0
    tie_weights: bool = True


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.record_attn = False      # flip on to store weights for visualisation
        self.last_attn = None

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.d_head).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin).type_as(v), apply_rope(k, cos, sin).type_as(v)
        if self.record_attn:
            y, w = attention(q, k, v, causal_mask(T, x.device))
            self.last_attn = w.detach().float().cpu()
        else:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                               dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.proj(y))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg.d_model, dropout=cfg.dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)   # pre-norm residual
        x = x + self.mlp(self.norm2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights:
            self.lm_head.weight = self.tok_emb.weight          # weight tying
        cos, sin = rope_angles(cfg.block_size, cfg.d_model // cfg.n_head, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("attn.proj.weight") or name.endswith("mlp.w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than block_size"
        x = self.drop(self.tok_emb(idx))
        for blk in self.blocks:
            x = blk(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())   # tied weights counted once

In [ ]:
if DEVICE == "cuda":   # ⚡ GPU: a bigger model, longer context
    cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=256, n_layer=6, n_head=4, d_model=256, dropout=0.1)
    BATCH, MAX_ITERS, PEAK_LR = 64, 2500, 1e-3
else:              # CPU: small but still produces recognisable text in ~6 min
    cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=128, n_layer=4, n_head=4, d_model=128, dropout=0.0)
    BATCH, MAX_ITERS, PEAK_LR = 32, 1500, 3e-3
if FAST:
    MAX_ITERS = 40
WARMUP, MIN_LR = max(1, MAX_ITERS // 15), PEAK_LR / 10
EVAL_EVERY, EVAL_ITERS = (20, 2) if FAST else (max(1, MAX_ITERS // 10), 20)

seed_everything()
model = GPT(cfg).to(DEVICE)
print(cfg)
print(f"parameters: {model.num_params() / 1e6:.2f} M")
for name, p in list(model.named_parameters())[:12]:
    print(f"  {name:32s} {tuple(p.shape)}")

### 🔮 Predict-then-run
Before training, what loss should a freshly initialised model have? *Hint:* with tiny initial weights the logits are almost all equal... Write down your guess, then run the cell.

In [ ]:
xb, yb = get_batch("train", BATCH, cfg.block_size)
with torch.no_grad():
    _, loss0 = model(xb, yb)
print(f"initial loss {loss0.item():.3f}   vs   ln(V) = {math.log(tok.vocab_size):.3f}")
assert abs(loss0.item() - math.log(tok.vocab_size)) < 0.3, "init is off: logits should start near-uniform"

It's ≈ $\ln V$: near-uniform predictions over 65 characters. (If you forget the small init, weight tying makes initial logits *huge* and the starting loss can exceed 100. Try it!)

### 🧪 Your Turn #6 — warmup + cosine learning-rate schedule
$$\eta(s) = \begin{cases}\eta_{max}\,\dfrac{s+1}{W} & s < W\\[6pt]
\eta_{min} + \tfrac12(\eta_{max}-\eta_{min})\big(1+\cos(\pi\,\tfrac{s-W}{S-W})\big) & W \le s < S\\[4pt]
\eta_{min} & s \ge S\end{cases}$$
**Why warmup?** Adam's second-moment estimates are garbage for the first few steps, and a large LR then can blow up the residual stream. **Why cosine?** Spending the end of training at a small LR lets the model settle into a minimum (lower final loss than a constant LR). Modern variants include warmup-stable-decay (WSD) schedules, which make it easy to continue training.

In [ ]:
def get_lr(step, warmup=WARMUP, max_steps=MAX_ITERS, max_lr=PEAK_LR, min_lr=MIN_LR):
    # TODO: linear warmup, then cosine decay to min_lr, then hold at min_lr
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def get_lr(step, warmup=WARMUP, max_steps=MAX_ITERS, max_lr=PEAK_LR, min_lr=MIN_LR):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    if step >= max_steps:
        return min_lr
    progress = (step - warmup) / max(1, max_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [ ]:
# ✔️ Checkpoint
assert abs(get_lr(0, 10, 100, 1.0, 0.1) - 0.1) < 1e-9
assert abs(get_lr(9, 10, 100, 1.0, 0.1) - 1.0) < 1e-9
assert abs(get_lr(55, 10, 100, 1.0, 0.1) - 0.55) < 1e-9      # halfway through the cosine
assert abs(get_lr(100, 10, 100, 1.0, 0.1) - 0.1) < 1e-9 and abs(get_lr(500, 10, 100, 1.0, 0.1) - 0.1) < 1e-9
steps = np.arange(MAX_ITERS)
plt.figure(figsize=(6, 2.8)); plt.plot(steps, [get_lr(s) for s in steps])
plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("warmup + cosine"); plt.show()
print("✔️ schedule correct")

### The training loop
Ingredients, each earning its keep:
* **AdamW** with decoupled weight decay (Loshchilov & Hutter) applied only to 2-D weight matrices, never to norm gains. $\beta_2 = 0.95$ is the LLM-standard choice (more responsive than 0.999 to sudden gradient changes).
* **Gradient clipping** to global norm 1.0: one bad batch can't kick us off a cliff.
* **Mixed precision (GPU only):** `torch.autocast` runs matmuls in bf16 (Ampere+) or fp16 (T4) while keeping master weights in fp32. fp16 has a narrow exponent range, so it needs a `GradScaler` to avoid gradient underflow; bf16 does not.

In [ ]:
decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1},
                               {"params": no_decay, "weight_decay": 0.0}],
                              lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8)

USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and AMP_DTYPE == torch.float16))
print("mixed precision:", f"on ({AMP_DTYPE})" if USE_AMP else "off (CPU)")

@torch.no_grad()
def estimate_loss(model, iters=EVAL_ITERS):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(iters):
            x, y = get_batch(split, BATCH, cfg.block_size)
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                _, l = model(x, y)
            losses.append(l.item())
        out[split] = float(np.mean(losses))
    model.train()
    return out

In [ ]:
history = {"step": [], "train": [], "val": [], "lr": [], "gnorm": [], "batch_loss": []}
snapshots = []   # (step, val loss, sample text) for the "watch it learn" animation below

@torch.no_grad()
def quick_sample(model, n=160, temperature=0.8, seed=0):
    """A short sample from a fixed seed, so snapshots are comparable across training."""
    was_training = model.training
    model.eval()
    g = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.tensor([tok.encode("ROMEO:\n")], dtype=torch.long, device=DEVICE)
    for _ in range(n):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        probs = (logits[:, -1, :].float().cpu() / temperature).softmax(-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1, generator=g).to(DEVICE)], dim=1)
    model.train(was_training)
    return tok.decode(idx[0].tolist())
seed_everything()
model.train()
t0 = time.time()
for step in range(MAX_ITERS + 1):
    if step % EVAL_EVERY == 0 or step == MAX_ITERS:
        l = estimate_loss(model)
        history["step"].append(step); history["train"].append(l["train"]); history["val"].append(l["val"])
        print(f"step {step:5d} | train {l['train']:.3f} | val {l['val']:.3f} | lr {get_lr(step):.2e} | {time.time() - t0:6.1f}s")
        snapshots.append((step, l["val"], quick_sample(model, n=60 if FAST else 160)))
    if step == MAX_ITERS:
        break
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    xb, yb = get_batch("train", BATCH, cfg.block_size)
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                       # so clipping sees true gradient norms
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    history["lr"].append(lr); history["gnorm"].append(gnorm.item()); history["batch_loss"].append(loss.item())
train_time = time.time() - t0
print(f"\n🐝 trained {MAX_ITERS} steps in {train_time / 60:.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(history["batch_loss"], alpha=0.3, label="batch loss")
axes[0].plot(history["step"], history["train"], "o-", label="train (est.)")
axes[0].plot(history["step"], history["val"], "s-", label="val (est.)")
axes[0].axhline(unigram_entropy, ls=":", color="gray", label="unigram baseline")
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy (nats)"); axes[0].legend(); axes[0].set_title("learning curves")
axes[1].plot(history["gnorm"], color=HIVE[4], alpha=0.7); axes[1].axhline(1.0, ls="--", color=HIVE[2], label="clip threshold")
axes[1].set_yscale("log"); axes[1].set_xlabel("step"); axes[1].set_title("gradient norm (pre-clip)"); axes[1].legend()
plt.tight_layout(); plt.show()
final_val = history["val"][-1]
print(f"final val loss {final_val:.3f} nats = {final_val / math.log(2):.2f} bits/char; perplexity {math.exp(final_val):.2f}")
if not FAST:
    assert final_val < 2.0, "val loss should be well below the unigram baseline after full training"

### 🎬 Watch it learn to write
Every time we evaluated, we also drew a sample **from the same random seed**. Press play: the same "dice rolls" go through an ever-better model, from random characters, to letter frequencies, to words, to speaker names and line breaks.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, (ax_l, ax_t) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.7]})
steps_s = [sn[0] for sn in snapshots]; vals_s = [sn[1] for sn in snapshots]
def draw(i):
    ax_l.clear(); ax_t.clear()
    ax_l.plot(steps_s, vals_s, "-", color="#D9D4C7")
    ax_l.plot(steps_s[:i + 1], vals_s[:i + 1], "o-", color=HIVE[2])
    ax_l.axhline(unigram_entropy, ls=":", color="gray"); ax_l.set_xlabel("step"); ax_l.set_ylabel("val loss")
    ax_l.set_title(f"step {steps_s[i]}  ·  val loss {vals_s[i]:.2f}")
    ax_t.axis("off")
    ax_t.text(0.0, 1.0, snapshots[i][2], family="monospace", fontsize=9, va="top", ha="left", wrap=True,
              transform=ax_t.transAxes)
    ax_t.set_title("sample (same seed every frame)")
anim = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=900)
plt.close(fig)
HTML(anim.to_jshtml())

> **🍯 Key takeaway:** Small init, warmup + cosine, AdamW (β₂ = 0.95), gradient clipping and mixed precision are the unglamorous hygiene that makes training just work.

## Part 7 · Generation and a look inside 🔍

**Autoregressive generation:** feed the context, take the logits at the last position, sample a token, append it, repeat. *Temperature* $\tau$ rescales logits ($p \propto e^{z/\tau}$): $\tau<1$ sharpens (safer, more repetitive), $\tau>1$ flattens (more creative, more typos). *Top-k* keeps only the $k$ most likely tokens. Lab 3.2 dives deep into sampling and makes this loop much faster with a **KV cache** (right now we recompute the whole prefix at every step!).

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=300, temperature=0.8, top_k=None, seed=0):
    model.eval()
    g = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        logits = logits[:, -1, :].float().cpu() / max(temperature, 1e-6)
        if top_k is not None:
            kth = torch.topk(logits, top_k).values[:, -1:]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        nxt = torch.multinomial(logits.softmax(-1), 1, generator=g).to(DEVICE)
        idx = torch.cat([idx, nxt], dim=1)
    return tok.decode(idx[0].tolist())

n_new = 60 if FAST else 400
for temp in [0.5, 0.9]:
    print(f"──────── temperature {temp} ────────")
    print(generate(model, "ROMEO:\n", n_new, temperature=temp, top_k=40))
    print()

In [ ]:
untrained = GPT(cfg).to(DEVICE)
print("──────── an UNTRAINED model, for contrast ────────")
print(generate(untrained, "ROMEO:\n", 120, temperature=1.0))

### Attention heatmaps of the trained model
We flip `record_attn` on, run a snippet through the model, and plot every head of every layer. Look for recognisable patterns:
* **previous-token heads** (a bright sub-diagonal): copy information from position $i-1$;
* heads that attend to the **start of the current word** or to the last **space/newline**;
* "sink" heads that dump attention on the first token when they have nothing useful to do (a well-documented phenomenon in LLMs, see "attention sinks").

In [ ]:
snippet = val_data[1000:1000 + 48].unsqueeze(0).to(DEVICE)
chars = [repr(c)[1:-1] for c in tok.decode(snippet[0].tolist())]
model.eval()
for blk in model.blocks:
    blk.attn.record_attn = True
with torch.no_grad():
    model(snippet)
for blk in model.blocks:
    blk.attn.record_attn = False
attn_maps = torch.stack([blk.attn.last_attn[0] for blk in model.blocks])   # (L, H, T, T)

L_, H_ = attn_maps.shape[:2]
fig, axes = plt.subplots(L_, H_, figsize=(2.6 * H_, 2.6 * L_))
for l in range(L_):
    for h in range(H_):
        ax = axes[l, h]
        ax.imshow(attn_maps[l, h].numpy(), cmap="YlOrBr", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        if h == 0: ax.set_ylabel(f"layer {l}")
        if l == 0: ax.set_title(f"head {h}")
plt.suptitle("attention weights for every head (rows = queries, cols = keys)"); plt.tight_layout(); plt.show()

In [ ]:
# Two simple "probes" for every head:
#   previous-token score: mean attention from position i to i-1
#   word-start score    : mean attention from position i to the most recent space/newline before i
T_ = attn_maps.shape[-1]
prev_score = attn_maps[..., torch.arange(1, T_), torch.arange(0, T_ - 1)].mean(-1)   # (L, H)
toks = snippet[0].tolist()
ws_ids = {tok.stoi[c] for c in " \n" if c in tok.stoi}
last_ws, cur = [], -1
for i, t in enumerate(toks):
    last_ws.append(cur)                      # most recent whitespace strictly before i
    if t in ws_ids: cur = i
q_idx = torch.tensor([i for i in range(T_) if 0 <= last_ws[i] < i - 1])   # skip cases identical to previous-token
k_idx = torch.tensor([last_ws[i] for i in q_idx.tolist()])
ws_score = attn_maps[..., q_idx, k_idx].mean(-1)
print("previous-token score per (layer, head):\n", prev_score.numpy().round(2))
print("word-start score per (layer, head):\n", ws_score.numpy().round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
for ax, score, name in [(axes[0], prev_score, "previous-token"), (axes[1], ws_score, "word-start")]:
    l_best, h_best = divmod(score.argmax().item(), H_)
    ax.imshow(attn_maps[l_best, h_best].numpy(), cmap="YlOrBr", vmin=0)
    ax.set_xticks(range(T_)); ax.set_xticklabels(chars, fontsize=6)
    ax.set_yticks(range(T_)); ax.set_yticklabels(chars, fontsize=6)
    ax.set_title(f"best {name} head: layer {l_best}, head {h_best} (score {score.max():.2f})")
    ax.set_xlabel("key (attended-to character)"); ax.set_ylabel("query character")
plt.tight_layout(); plt.show()

A pure previous-token head shows a clean sub-diagonal. A **word-start** head instead makes every character of a word look back at the space (or newline) just before the word: a cheap way to know "which word am I in, and how far into it". Your two panels may show the same head or different ones, depending on the run.

In models with ≥2 layers, previous-token heads in early layers often team up with later heads to form **induction heads** ("I saw `A B` before; now I see `A` again, so predict `B`"), a mechanism behind in-context learning (Olsson et al., 2022). Character-level Shakespeare has plenty of repeated names, so try feeding a snippet containing a name twice and look at the later layers!

> **🍯 Key takeaway:** Generation is just a loop of forward → sample → append, and trained heads develop readable jobs (previous-token, word-start) that you can find with simple probes.

## 🏆 Boss Level
1. **Grouped-Query Attention (GQA).** Give the model `n_kv_head < n_head` key/value heads (each K/V head shared by `n_head // n_kv_head` query heads; `repeat_interleave` the K/V tensors before attention). Measure parameter count, the KV-cache size per token, and val loss for `n_kv_head ∈ {4, 2, 1}` (1 = multi-query attention). This is what Llama 2 70B / Llama 3 do to shrink the KV cache.
2. **Ablation study.** Swap one modern component back to its 2017 version at a time: LayerNorm instead of RMSNorm, GELU MLP instead of SwiGLU, learned absolute positions instead of RoPE, post-norm instead of pre-norm, untied embeddings. Train each for the same number of steps and make a bar chart of final val loss. Which change hurts most at this tiny scale? (Expect small differences and run 2–3 seeds; noise is real.)
3. **Length extrapolation.** Train with `block_size=64`, then evaluate val loss at contexts of 64, 128, 256 (rebuild the RoPE tables longer). Then implement *Position Interpolation* (divide positions by 4) and/or raise `rope_base`, and see how much of the extrapolation gap you can close.

## 🍯 Key Takeaways
- **Attention** = softmax$(QK^\top/\sqrt{d})V$: a differentiable lookup. The $\sqrt{d}$ prevents softmax saturation; the **causal mask** gives you T training examples per window.
- **Multi-head** attention costs nothing extra: it's a reshape that lets different heads specialise.
- **RoPE** encodes *relative* position by rotating q and k, with no parameters, and its frequencies are the knob that long-context extensions turn.
- The **Llama-style block** (pre-norm, RMSNorm, SwiGLU, no biases, optional tied embeddings) is the backbone of most open LLMs.
- **Training hygiene** (init scale, AdamW β₂ = 0.95, warmup + cosine, clipping, bf16/fp16 autocast) matters as much as architecture at small scale.
- A ~1M-parameter model goes from 4.17 nats (uniform) to about 1.5 nats in minutes on a CPU and learns speakers, line breaks and English words.

## 🎤 Explain it to a friend
Explain to a non-programmer how your model writes Shakespeare **one character at a time**: what it "looks at" before choosing each letter, why it has several "heads" that look at different things (like the previous letter, or the start of the current word), and why it gets better simply by guessing the next letter of real plays millions of times. Bonus: explain what "temperature" does using a dice analogy.

## 💼 Interview corner
1. **Why do we divide attention scores by √d_k?**
   <details><summary>Answer</summary>If q and k have independent unit-variance entries, q·k has variance d_k. Large logits push softmax toward one-hot, where its gradients vanish and training stalls. Dividing by √d_k keeps the logits at unit variance whatever the head size.</details>
2. **What does RoPE give you compared with learned absolute position embeddings?**
   <details><summary>Answer</summary>RoPE rotates q and k by angles proportional to position, so q·k depends only on the relative offset m − n. It adds no parameters, applies inside every attention layer rather than only at the input, and its frequencies can be rescaled (Position Interpolation, NTK-aware scaling, YaRN) to extend the context window after training. Learned absolute embeddings simply have no entries for positions they never saw.</details>
3. **Why do modern LLMs use pre-norm rather than post-norm, and what's the catch?**
   <details><summary>Answer</summary>Pre-norm (x + f(Norm(x))) keeps a clean identity path through the residual stream, so gradients flow through deep stacks and training is stable with little warmup tuning. The catch: the residual stream's magnitude can grow with depth, which is why a final norm is applied before the LM head, and why residual-projection init is often scaled down (e.g. by 1/√(2L)).</details>

## 🐝 Fun fact
GPT-2's tokenizer has exactly **50,257** tokens: 256 byte-level base tokens, 50,000 learned BPE merges, and one special `<|endoftext|>` token.

## 📚 Further reading
- Vaswani et al., *Attention Is All You Need* (2017).
- Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2, 2019).
- Su et al., *RoFormer: Enhanced Transformer with Rotary Position Embedding* (2021).
- Zhang & Sennrich, *Root Mean Square Layer Normalization* (2019).
- Shazeer, *GLU Variants Improve Transformer* (2020).
- Xiong et al., *On Layer Normalization in the Transformer Architecture* (2020) — pre-norm vs post-norm.
- Press & Wolf, *Using the Output Embedding to Improve Language Models* (2017) — weight tying.
- Loshchilov & Hutter, *Decoupled Weight Decay Regularization* (2019) — AdamW.
- Touvron et al., *LLaMA: Open and Efficient Foundation Language Models* (2023).
- Ainslie et al., *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints* (2023).
- Olsson et al., *In-context Learning and Induction Heads* (2022).
- Andrej Karpathy, *nanoGPT* (GitHub) and the video "Let's build GPT: from scratch, in code, spelled out" (2023).